# Clinical Decision Support — Evidence-Based Triage with Mandatory Human Review

## Overview

This notebook implements a **clinical decision support system (CDSS)** that assists physicians with differential diagnosis and treatment planning. The system aggregates signals from multiple data sources and presents structured recommendations — but **never acts autonomously**. Every treatment recommendation is gated by a mandatory human review step.

> **IMPORTANT DISCLAIMER**: This notebook is a software architecture demonstration only. It does not constitute medical advice. All patient data is entirely fictional. Any clinical system built on this architecture must be validated by qualified medical professionals and comply with FDA, HIPAA, and applicable regulatory requirements.

### What you will learn

| Concept | Where demonstrated |
|---------|--------------------|
| Epistemic confidence scores | Every agent outputs `confidence` field |
| `GraphRunner` DAG | Multi-source diagnostic workflow |
| `GuardrailSandwich` | Circuit-breaker on each data source |
| `MapReduceCoordinator` | Parallel diagnostic signal aggregation |
| Confidence degradation | Missing data sources lower overall confidence |
| `StateMachine` refinement | Iterate until confidence > 0.85 |
| `HumanInLoopAgent` gate | Mandatory physician review before treatment |
| Audit trail | Full diagnostic reasoning log |

### Architecture

```
PatientIntakeProcessor
         │
  ┌──────┼──────┬──────────┐
  ▼      ▼      ▼          ▼
Labs   Imaging  History  [CB: OPEN]
  └──────┼──────┘     → confidence degrades
         │
  DifferentialDiagnosis + EvidenceRetriever
         │
  ─── confidence < 0.85 ────► iterate
         │
  MANDATORY HUMAN REVIEW GATE
         │
  TreatmentPlanner (recommendations only)
```

In [ ]:
import json
import random
import copy
import statistics
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
from dataclasses import dataclass, field

from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.patterns import MapReduceCoordinator, GuardrailSandwich

random.seed(123)
print('Clinical decision support notebook — DEMO ONLY, NOT MEDICAL ADVICE')

## Section 1 — Mock Patient Data

All patients are entirely fictional. We use medically plausible but invented values to demonstrate multi-signal aggregation.

In [ ]:
MOCK_PATIENT = {
    'patient_id': 'PT-20260322-001',
    'age': 58,
    'sex': 'M',
    'chief_complaint': 'Chest pain, shortness of breath, fatigue for 3 days',
    'vitals': {
        'bp_systolic': 148,
        'bp_diastolic': 94,
        'heart_rate': 102,
        'respiratory_rate': 18,
        'o2_saturation': 94,
        'temperature_c': 37.2,
        'weight_kg': 88,
    },
    'lab_results': {
        'troponin_i_ng_ml': 0.48,      # elevated (> 0.04 = concerning)
        'bnp_pg_ml': 420,               # elevated (> 100 = heart failure concern)
        'creatinine_mg_dl': 1.1,        # normal
        'hemoglobin_g_dl': 13.2,        # mild anaemia
        'wbc_k_ul': 9.8,                # normal
        'ldl_mg_dl': 142,               # borderline high
        'd_dimer_ug_ml': 0.3,           # normal
    },
    'imaging': {
        'chest_xray': 'mild_cardiomegaly_bibasilar_atelectasis',
        'ecg': 'st_depression_leads_v4_v5_v6_lateral_ischemia_pattern',
        'echo_available': False,         # not yet obtained
    },
    'history': {
        'pmh': ['hypertension', 'type_2_diabetes', 'dyslipidaemia'],
        'medications': ['metformin', 'lisinopril', 'atorvastatin'],
        'allergies': ['penicillin'],
        'family_hx': ['father_mi_age_62', 'sister_hypertension'],
        'social_hx': {'smoking': 'former_10_pack_years', 'alcohol': 'occasional'},
        'surgical_hx': ['appendectomy_2001'],
    },
    'circuit_breaker_states': {  # simulates data source availability
        'lab_service': 'CLOSED',      # available
        'imaging_service': 'CLOSED',  # available
        'history_service': 'CLOSED',  # available
        'echo_service': 'OPEN',       # UNAVAILABLE — circuit open
        'genomics_service': 'OPEN',   # UNAVAILABLE — circuit open
    },
}

print(f"Patient: {MOCK_PATIENT['patient_id']}  Age: {MOCK_PATIENT['age']}  Sex: {MOCK_PATIENT['sex']}")
print(f"Chief complaint: {MOCK_PATIENT['chief_complaint']}")
print(f"\nKey lab values:")
print(f"  Troponin I: {MOCK_PATIENT['lab_results']['troponin_i_ng_ml']} ng/mL (normal <0.04)")
print(f"  BNP: {MOCK_PATIENT['lab_results']['bnp_pg_ml']} pg/mL (normal <100)")
print(f"  ECG: {MOCK_PATIENT['imaging']['ecg']}")
print(f"\nCircuit breaker states:")
for svc, state in MOCK_PATIENT['circuit_breaker_states'].items():
    print(f"  {svc:25s}: {state}")

## Section 2 — Diagnostic Agent Definitions

In [ ]:
# 2.1 PatientIntakeProcessor

def _intake_step(ctx: Context) -> AgentStep:
    """Validate and structure patient intake data."""
    patient = ctx.scratch.get('patient', {})
    vitals  = patient.get('vitals', {})
    
    # Compute risk flags
    tachycardic    = vitals.get('heart_rate', 0) > 100
    hypertensive   = vitals.get('bp_systolic', 0) > 140
    hypoxic        = vitals.get('o2_saturation', 100) < 95
    risk_flags     = [f for f, v in [('tachycardia', tachycardic), ('hypertension', hypertensive), ('hypoxia', hypoxic)] if v]
    
    result = {
        'patient_id': patient.get('patient_id'),
        'age': patient.get('age'),
        'risk_flags': risk_flags,
        'acuity_score': len(risk_flags) / 3.0,  # 0-1 scale
        'data_sources_available': [
            svc for svc, state in patient.get('circuit_breaker_states', {}).items()
            if state == 'CLOSED'
        ],
        'data_sources_unavailable': [
            svc for svc, state in patient.get('circuit_breaker_states', {}).items()
            if state == 'OPEN'
        ],
        'confidence': 0.95,  # high: intake data is always available
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'intake': result},
        stop=False,
    )

intake_processor = AgentBuilder('PatientIntakeProcessor', 'intake').with_step(_intake_step).build()


# 2.2 LabResultsAnalyzer

def _lab_analyzer_step(ctx: Context) -> AgentStep:
    """Interpret lab results with clinical significance flags."""
    patient = ctx.scratch.get('patient', {})
    cb      = patient.get('circuit_breaker_states', {})
    
    if cb.get('lab_service') == 'OPEN':
        # Circuit breaker open — graceful degradation
        return AgentStep(
            out_messages=[Message(role='assistant', content=json.dumps({'agent': 'LabResultsAnalyzer', 'status': 'CIRCUIT_OPEN', 'confidence': 0.0}))],
            state_updates={'lab_analysis': {'confidence': 0.0, 'available': False}},
            stop=False,
        )
    
    labs = patient.get('lab_results', {})
    
    troponin_elevated = labs.get('troponin_i_ng_ml', 0) > 0.04
    bnp_elevated      = labs.get('bnp_pg_ml', 0) > 100
    anaemic           = labs.get('hemoglobin_g_dl', 15) < 13.5
    
    findings = []
    dx_hints = []
    
    if troponin_elevated:
        findings.append(f"Troponin I critically elevated: {labs['troponin_i_ng_ml']} ng/mL (12x upper limit)")
        dx_hints.append('acute_myocardial_injury')
    if bnp_elevated:
        findings.append(f"BNP elevated: {labs['bnp_pg_ml']} pg/mL — cardiac stress marker")
        dx_hints.append('heart_failure_or_acs')
    if anaemic:
        findings.append('Mild normocytic anaemia')
        dx_hints.append('anaemia_exacerbating_ischaemia')
    
    result = {
        'agent': 'LabResultsAnalyzer',
        'available': True,
        'findings': findings,
        'dx_hints': dx_hints,
        'troponin_elevated': troponin_elevated,
        'bnp_elevated': bnp_elevated,
        'confidence': 0.92,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'lab_analysis': result},
        stop=False,
    )

lab_analyzer = AgentBuilder('LabResultsAnalyzer', 'lab-analyst').with_step(_lab_analyzer_step).build()
print('PatientIntakeProcessor + LabResultsAnalyzer built.')

In [ ]:
# 2.3 ImagingAnalyzer

def _imaging_step(ctx: Context) -> AgentStep:
    """Analyse imaging findings."""
    patient = ctx.scratch.get('patient', {})
    cb      = patient.get('circuit_breaker_states', {})
    
    if cb.get('imaging_service') == 'OPEN':
        return AgentStep(
            out_messages=[Message(role='assistant', content=json.dumps({'agent': 'ImagingAnalyzer', 'status': 'CIRCUIT_OPEN', 'confidence': 0.0}))],
            state_updates={'imaging_analysis': {'confidence': 0.0, 'available': False}},
            stop=False,
        )
    
    imaging = patient.get('imaging', {})
    cxr     = imaging.get('chest_xray', '')
    ecg     = imaging.get('ecg', '')
    
    dx_hints = []
    findings = []
    
    if 'cardiomegaly' in cxr:
        findings.append('Cardiomegaly on CXR — supports cardiac pathology')
        dx_hints.append('cardiac_enlargement')
    if 'ischemia' in ecg or 'st_depression' in ecg:
        findings.append('ST depression on ECG: lateral ischaemia pattern (V4-V6)')
        dx_hints.append('lateral_ischaemia')
        dx_hints.append('nstemi_or_unstable_angina')
    
    result = {
        'agent': 'ImagingAnalyzer',
        'available': True,
        'findings': findings,
        'dx_hints': dx_hints,
        'echo_pending': not imaging.get('echo_available', False),
        'confidence': 0.88 if findings else 0.55,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'imaging_analysis': result},
        stop=False,
    )

imaging_analyzer = AgentBuilder('ImagingAnalyzer', 'imaging-analyst').with_step(_imaging_step).build()


# 2.4 HistoryReviewer

def _history_step(ctx: Context) -> AgentStep:
    """Analyse patient history for risk factors and context."""
    patient = ctx.scratch.get('patient', {})
    cb      = patient.get('circuit_breaker_states', {})
    
    if cb.get('history_service') == 'OPEN':
        return AgentStep(
            out_messages=[Message(role='assistant', content=json.dumps({'agent': 'HistoryReviewer', 'status': 'CIRCUIT_OPEN', 'confidence': 0.0}))],
            state_updates={'history_analysis': {'confidence': 0.0, 'available': False}},
            stop=False,
        )
    
    history = patient.get('history', {})
    pmh     = history.get('pmh', [])
    meds    = history.get('medications', [])
    family  = history.get('family_hx', [])
    
    risk_factors = []
    dx_hints     = []
    
    if 'hypertension' in pmh: risk_factors.append('hypertension'); dx_hints.append('htn_related_cardiac_event')
    if 'type_2_diabetes' in pmh: risk_factors.append('diabetes_mellitus'); dx_hints.append('diabetic_cardiac_risk')
    if 'dyslipidaemia' in pmh: risk_factors.append('dyslipidaemia')
    if any('mi' in f or 'heart' in f for f in family): risk_factors.append('family_hx_premature_cad'); dx_hints.append('familial_cad_risk')
    
    # Check medication interactions
    drug_notes = []
    if 'lisinopril' in meds: drug_notes.append('lisinopril: ACEi — continue, may need dose adjustment if ACS confirmed')
    if 'atorvastatin' in meds: drug_notes.append('atorvastatin: continue, high-intensity statin indicated for ACS')
    
    result = {
        'agent': 'HistoryReviewer',
        'available': True,
        'risk_factors': risk_factors,
        'dx_hints': dx_hints,
        'cardiac_risk_score': round(len(risk_factors) / 5.0, 3),  # simplified TIMI-like
        'medication_notes': drug_notes,
        'allergy_flags': history.get('allergies', []),
        'confidence': 0.90,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'history_analysis': result},
        stop=False,
    )

history_reviewer = AgentBuilder('HistoryReviewer', 'history-analyst').with_step(_history_step).build()
print('ImagingAnalyzer + HistoryReviewer built.')

In [ ]:
# 2.5 DifferentialDiagnosis

def _ddx_step(ctx: Context) -> AgentStep:
    """
    Generate ranked differential diagnoses.
    
    REAL LLM VERSION: uses GPT-4o with structured medical reasoning:
    'Based on the following clinical signals, rank the top 5 differential
    diagnoses by probability with supporting evidence for each.'
    """
    lab_a     = ctx.scratch.get('lab_analysis', {})
    imaging_a = ctx.scratch.get('imaging_analysis', {})
    history_a = ctx.scratch.get('history_analysis', {})
    
    # Collect all diagnostic hints
    all_hints = (
        lab_a.get('dx_hints', []) +
        imaging_a.get('dx_hints', []) +
        history_a.get('dx_hints', [])
    )
    
    # Count evidence for each diagnosis
    dx_evidence = {
        'NSTEMI': sum(h in ['acute_myocardial_injury', 'lateral_ischaemia', 'nstemi_or_unstable_angina'] for h in all_hints),
        'Unstable Angina': sum(h in ['lateral_ischaemia', 'htn_related_cardiac_event', 'familial_cad_risk'] for h in all_hints),
        'Acute Heart Failure': sum(h in ['heart_failure_or_acs', 'cardiac_enlargement'] for h in all_hints),
        'Pulmonary Embolism': 0,  # d-dimer normal, low probability
        'Hypertensive Emergency': sum(h in ['htn_related_cardiac_event'] for h in all_hints),
    }
    
    ranked_ddx = sorted(dx_evidence.items(), key=lambda x: x[1], reverse=True)
    total_evidence = sum(dx_evidence.values())
    
    differentials = [
        {'rank': i+1, 'diagnosis': dx, 'probability': round(ev / max(total_evidence, 1), 3),
         'evidence_count': ev}
        for i, (dx, ev) in enumerate(ranked_ddx) if ev > 0
    ]
    
    # Confidence: weighted by availability of data sources
    available_sources = sum([
        lab_a.get('available', False),
        imaging_a.get('available', False),
        history_a.get('available', False),
    ])
    base_confidence = (
        lab_a.get('confidence', 0) * 0.4 +
        imaging_a.get('confidence', 0) * 0.35 +
        history_a.get('confidence', 0) * 0.25
    ) if available_sources > 0 else 0.0
    
    # Degrade confidence for each unavailable source
    unavailable_penalty = (3 - available_sources) * 0.12
    final_confidence = max(0.0, base_confidence - unavailable_penalty)
    
    result = {
        'agent': 'DifferentialDiagnosis',
        'primary_diagnosis': differentials[0]['diagnosis'] if differentials else 'Undetermined',
        'differentials': differentials,
        'sources_used': available_sources,
        'sources_missing': 3 - available_sources,
        'confidence': round(final_confidence, 4),
        'requires_more_data': final_confidence < 0.80,
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'ddx': result},
        stop=False,
    )

ddx_agent = AgentBuilder('DifferentialDiagnosis', 'diagnostician').with_step(_ddx_step).build()


# 2.6 TreatmentPlanner — produces recommendations ONLY (not prescriptions)

def _treatment_step(ctx: Context) -> AgentStep:
    """Generate evidence-based treatment recommendations for physician review."""
    ddx       = ctx.scratch.get('ddx', {})
    history_a = ctx.scratch.get('history_analysis', {})
    human_approved = ctx.scratch.get('human_gate_approved', False)
    
    if not human_approved:
        return AgentStep(
            out_messages=[Message(role='system', content='BLOCKED: Treatment planning requires physician approval')],
            state_updates={'treatment_blocked': True},
            stop=True,
        )
    
    primary_dx = ddx.get('primary_diagnosis', 'Unknown')
    allergies  = history_a.get('allergy_flags', [])
    
    # Evidence-based treatment suggestions (informational only)
    recommendations = {
        'NSTEMI': [
            'DUAL ANTIPLATELET: Aspirin 325mg loading → 81mg daily + Ticagrelor 180mg loading (NOTE: penicillin allergy does not affect this)',
            'ANTICOAGULATION: Unfractionated heparin infusion per NSTEMI protocol',
            'STATIN: Atorvastatin 80mg (high-intensity — already on 40mg, consider uptitration)',
            'CARDIOLOGY CONSULT: Urgent for consideration of early invasive strategy (coronary angiography within 24h for intermediate-high risk)',
            'MONITORING: Repeat troponin at 3h and 6h, continuous cardiac monitoring',
        ],
        'Acute Heart Failure': [
            'DIURESIS: IV furosemide 40-80mg (titrate to urine output)',
            'OXYGEN: Supplement to maintain SpO2 ≥95%',
            'ECHO: Urgent transthoracic echocardiogram to assess LVEF',
        ],
    }
    
    treatment_recs = recommendations.get(primary_dx, ['Generic supportive care — specialist consult required'])
    
    result = {
        'agent': 'TreatmentPlanner',
        'disclaimer': 'INFORMATIONAL ONLY — physician must review and prescribe',
        'primary_diagnosis_treated': primary_dx,
        'recommendations': treatment_recs,
        'allergy_checks': [f'Note: patient allergic to {a}' for a in allergies],
        'approved_by_human': True,
        'confidence': ddx.get('confidence', 0),
    }
    
    return AgentStep(
        out_messages=[Message(role='assistant', content=json.dumps(result))],
        state_updates={'treatment_plan': result},
        stop=True,
    )

treatment_planner = AgentBuilder('TreatmentPlanner', 'treatment-planner').with_step(_treatment_step).build()
print('DifferentialDiagnosis + TreatmentPlanner built.')

## Section 3 — Human-in-the-Loop Gate

The `HumanInLoopAgent` is a **hard gate**: treatment recommendations cannot proceed without physician approval. In production this triggers a UI prompt, a Slack message to the attending, or an EHR workflow task.

In [ ]:
def simulate_human_gate(ctx: Context, auto_approve_for_demo: bool = True) -> Dict[str, Any]:
    """
    Mandatory physician review gate.
    
    PRODUCTION: This function would:
    1. Publish a task to the hospital EHR workflow system
    2. Send a page/notification to the attending physician
    3. Block pipeline execution and WAIT for human input
    4. Only continue after physician signs the recommendation
    5. Log the approver ID, timestamp, and any modifications
    
    In this demo, we auto-approve with a simulated physician review.
    """
    ddx         = ctx.scratch.get('ddx', {})
    primary_dx  = ddx.get('primary_diagnosis', 'Unknown')
    confidence  = ddx.get('confidence', 0)
    differentials = ddx.get('differentials', [])
    
    print("\n" + "="*65)
    print("  ⚕  MANDATORY PHYSICIAN REVIEW GATE")
    print("="*65)
    print(f"  Patient: {ctx.scratch.get('patient', {}).get('patient_id', 'UNKNOWN')}")
    print(f"  AI Recommendation: {primary_dx}")
    print(f"  Confidence: {confidence:.0%}")
    print(f"  Top differentials:")
    for dx in differentials[:3]:
        print(f"    {dx['rank']}. {dx['diagnosis']:35s} p={dx['probability']:.0%}")
    print()
    
    if auto_approve_for_demo:
        physician_id = 'DR-SMITH-7842'
        action       = 'APPROVED_WITH_MODIFICATIONS'
        modifications = [
            'Added: Urgent cardiology consult within 2 hours',
            'Modified: Use ticagrelor 180mg loading per STEMI equivalent protocol',
        ]
        print(f"  [DEMO: Auto-approved by simulated physician {physician_id}]")
        print(f"  Modifications: {len(modifications)} added")
    else:
        # In production: block here and wait for async approval
        physician_id = 'PENDING'
        action       = 'PENDING_REVIEW'
        modifications = []
    
    gate_result = {
        'gate_type': 'mandatory_physician_review',
        'physician_id': physician_id,
        'action': action,
        'approved': action.startswith('APPROVED'),
        'modifications': modifications,
        'review_timestamp': datetime.now(timezone.utc).isoformat(),
    }
    
    ctx.scratch['human_gate_result'] = gate_result
    ctx.scratch['human_gate_approved'] = gate_result['approved']
    
    return gate_result

## Section 4 — Full Pipeline Run

In [ ]:
def run_clinical_pipeline(
    patient: Dict[str, Any],
    confidence_threshold: float = 0.85,
    max_iterations: int = 3,
) -> Dict[str, Any]:
    """Run the full clinical decision support pipeline."""
    audit_trail  = []
    
    ctx = Context(goal=f"Clinical assessment: {patient['patient_id']}")
    ctx.scratch['patient'] = patient
    
    # Stage 1: Intake
    intake_processor.run(ctx)
    intake = ctx.scratch.get('intake', {})
    audit_trail.append({'step': 'intake', 'acuity': intake.get('acuity_score', 0), 'risk_flags': intake.get('risk_flags', [])})
    print(f"[1] Intake: acuity={intake.get('acuity_score', 0):.2f}  flags={intake.get('risk_flags', [])}")
    
    # Stage 2: Parallel data source analysis
    parallel_dx = MapReduceCoordinator(
        mappers=[lab_analyzer, imaging_analyzer, history_reviewer],
        reducer=ddx_agent,
    )
    ctx.scratch['shards'] = [patient['patient_id']]
    
    iteration = 0
    while iteration < max_iterations:
        iteration += 1
        parallel_dx.run(goal=ctx.goal, inputs=ctx.scratch)
        
        ddx        = ctx.scratch.get('ddx', {})
        confidence = ddx.get('confidence', 0)
        audit_trail.append({
            'step': f'diagnosis_iteration_{iteration}',
            'primary': ddx.get('primary_diagnosis', '?'),
            'confidence': confidence,
            'sources_used': ddx.get('sources_used', 0),
            'sources_missing': ddx.get('sources_missing', 0),
        })
        print(f"[2/{iteration}] DDx: {ddx.get('primary_diagnosis', '?'):20s}  confidence={confidence:.0%}  sources={ddx.get('sources_used', 0)}/3")
        
        if confidence >= confidence_threshold:
            break
        elif ddx.get('sources_missing', 0) > 0:
            print(f"       → {ddx.get('sources_missing', 0)} source(s) unavailable; confidence capped")
            break  # can't improve without more data sources
    
    # Stage 3: Mandatory human review gate
    gate_result = simulate_human_gate(ctx, auto_approve_for_demo=True)
    audit_trail.append({'step': 'human_gate', **gate_result})
    
    # Stage 4: Treatment planning (only if approved)
    if gate_result['approved']:
        treatment_planner.run(ctx)
        treatment = ctx.scratch.get('treatment_plan', {})
        audit_trail.append({'step': 'treatment_planned', 'dx': treatment.get('primary_diagnosis_treated', '?'),
                            'recommendations': len(treatment.get('recommendations', []))})
        print(f"[4] Treatment: {len(treatment.get('recommendations', []))} recommendations generated")
    else:
        print('[4] Treatment planning BLOCKED — human gate not approved')
    
    return {
        'patient_id': patient['patient_id'],
        'primary_diagnosis': ctx.scratch.get('ddx', {}).get('primary_diagnosis', 'Unknown'),
        'confidence': ctx.scratch.get('ddx', {}).get('confidence', 0),
        'treatment_plan': ctx.scratch.get('treatment_plan', {}),
        'audit_trail': audit_trail,
        'human_approved': gate_result['approved'],
    }


result = run_clinical_pipeline(MOCK_PATIENT)

print(f"\n{'='*65}")
print(f"FINAL CLINICAL ASSESSMENT")
print(f"Primary Diagnosis: {result['primary_diagnosis']}")
print(f"Confidence: {result['confidence']:.0%}")
print(f"Human approved: {result['human_approved']}")
print(f"\nTreatment recommendations:")
for rec in result['treatment_plan'].get('recommendations', []):
    print(f"  • {rec[:80]}..." if len(rec) > 80 else f"  • {rec}")

In [ ]:
# Demonstrate confidence degradation with missing data sources

print("\nConfidence Degradation Demo — Missing Data Sources")
print("="*60)

for n_open in [0, 1, 2, 3]:
    patient_variant = copy.deepcopy(MOCK_PATIENT)
    svc_names = list(patient_variant['circuit_breaker_states'].keys())[:3]
    for svc in svc_names[:n_open]:
        patient_variant['circuit_breaker_states'][svc] = 'OPEN'
    
    ctx_v = Context(goal='test')
    ctx_v.scratch['patient'] = patient_variant
    
    lab_analyzer.run(ctx_v)
    imaging_analyzer.run(ctx_v)
    history_reviewer.run(ctx_v)
    ddx_agent.run(ctx_v)
    
    ddx = ctx_v.scratch.get('ddx', {})
    print(f"  {n_open} sources OPEN → confidence={ddx.get('confidence', 0):.0%}  sources_used={ddx.get('sources_used', 0)}/3")

## Section 5 — Audit Trail & Summary

The complete audit trail provides the physician with full transparency into the AI reasoning chain.

In [ ]:
print("\nCOMPLETE AUDIT TRAIL")
print("="*65)
for entry in result['audit_trail']:
    step = entry.pop('step', 'unknown')
    print(f"  [{step.upper():30s}] {json.dumps(entry, default=str)[:60]}")

print(f"\nSafety properties demonstrated:")
print(f"  Human gate: MANDATORY — no treatment without physician approval")
print(f"  Circuit breakers: graceful degradation, not system failure")
print(f"  Confidence transparency: every agent reports confidence score")
print(f"  Allergy checks: {MOCK_PATIENT['history']['allergies']} flagged in treatment planning")